# Correlated Disorder Structure Generation

Workflow for generating and saving 2D correlated-disorder particle structures.

Steps:
1. Generate a correlated-disorder structure via iterative centroidal relaxation
2. Visualize and inspect the result
3. Save positions to CSV
4. (Optional) Apply density reduction to create mixed ordered/disordered areas

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from phd_tools.structure_gen import (
    make_correlated_disorder,
    define_area_distance_and_radius,
    create_areas,
    select_area_points,
    load_positions_files,
)

## 1. Generate correlated disorder structure

The algorithm starts from N random points and iteratively replaces each point with the
centroid of its Voronoi cell. More iterations → more uniform nearest-neighbour distance.

Key parameters:
- `N` — number of particles
- `mesh_precision` — grid density for the centroidal approximation (higher = more accurate, slower)
- `iterations` — number of relaxation steps
- `perodic_boundaries` — mirror boundary particles to suppress edge effects

In [ ]:
N               = 1000
mesh_precision  = 10
iterations      = 50
max_xy          = np.asarray([1.0, 1.0])  # physical extent of the domain

structure = make_correlated_disorder(
    N,
    mesh_precision,
    iterations,
    perodic_boundaries=True,
    max_xy=max_xy,
)

## 2. Visualize and inspect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Particle positions
axes[0].plot(structure[:, 0], structure[:, 1], '.', ms=4)
axes[0].set_aspect('equal')
axes[0].set_title(f'Structure ({len(structure)} particles)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Nearest-neighbour distance distribution
tree = cKDTree(structure)
d, _ = tree.query(structure, k=2)
nn_distances = d[:, 1]
axes[1].hist(nn_distances, bins=30)
axes[1].set_title('Nearest-neighbour distance distribution')
axes[1].set_xlabel('Distance')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Mean NN distance: {nn_distances.mean():.4f}')
print(f'Std  NN distance: {nn_distances.std():.4f}')

## 3. Save structure to CSV

In [ ]:
output_file = Path(f'../data/generated_structure_N{N}_mesh{mesh_precision}_iter{iterations}.csv')
np.savetxt(str(output_file), structure, delimiter=',')
print(f'Saved to {output_file}')

## 4. (Optional) Density reduction

Selectively removes particles inside (or outside) a set of disordered circular areas.
The areas themselves are placed using correlated disorder, so both the particles
and the removal pattern share the same statistical character.

Parameters:
- `area_distance` — target average distance between area centres (same units as the structure)
- `density_reduction_proportion` — fraction of the total surface covered by the areas
- `keep_inside=True` — keep particles inside the areas (remove outside); `False` inverts this

In [ ]:
# Load an existing structure (or use `structure` from above)
# structure = load_positions_files('my_structure', '../data')

area_distance              = 0.15   # adjust to your structure scale
density_reduction_proportion = 0.5  # 50 % of surface covered

areas_poly, d_ave, r = define_area_distance_and_radius(
    structure,
    area_distance,
    density_reduction_proportion=density_reduction_proportion,
    keep_inside=True,
)

reduced_structure = select_area_points(areas_poly, structure, inside=True)
print(f'Particles after reduction: {len(reduced_structure)} (from {len(structure)})')

plt.figure(figsize=(7, 7))
plt.plot(reduced_structure[:, 0], reduced_structure[:, 1], '.', ms=3)
plt.axis('equal')
plt.title('Structure after density reduction')
plt.show()